# 00.2 NumPy 核心

这份 notebook 只讲后续 `PyTorch` 必需的 `NumPy` 内容，不追求面面俱到。  

本节最重要的两个习惯

1. 一看到数组操作就先看 `shape`
2. 尽量用向量化思维

## 学习目标

学完后你应该能

1. 创建并检查 `ndarray`
2. 熟练使用索引、切片、布尔索引
3. broadcasting 的基本规则
4. 把循环改写为向量化表达式
5. 正确处理 `reshape`、增加维度和转置
6. 把这些能力迁移到后面的 `PyTorch Tensor`

In [ ]:
import numpy as np

## 1. `ndarray` 基础

`NumPy` 的核心对象是 `ndarray`。很多 `PyTorch Tensor` 的操作风格和它非常接近。  

重点术语

- 形状
- 维度数
- 数据类型

In [ ]:
a = np.array([1, 2, 3], dtype=np.float32)
b = np.zeros((2, 3))
c = np.arange(12).reshape(3, 4)

print("a =", a)
print("a.shape / a 的形状 =", a.shape)
print("a.dtype / a 的类型 =", a.dtype)
print("a.ndim / a 的维度数 =", a.ndim)
print()
print("b =")
print(b)
print("b.shape =", b.shape)
print()
print("c =")
print(c)
print("c.shape =", c.shape)

要特别敏感的不是数值本身，而是这些结构信息：  

- `shape`：每个维度长度
- `ndim`：维度总数（total number of dimensions）
- `dtype`：元素类型（element type）

很多模型报错，第一步都应该先看 `shape` 和 `dtype`。  


In [ ]:
# 练习 1
# 目标
# 1. 创建从 0 到 11 的 float32 数组
# 2. reshape 成 (3, 4)
# 3. 打印 shape、dtype、最后一列

arr = None

# print(arr)
# print(arr.shape)
# print(arr.dtype)
# print(arr[:, -1])

In [ ]:
# 练习 1 参考答案

arr = np.arange(12, dtype=np.float32).reshape(3, 4)
print(arr)
print(arr.shape)
print(arr.dtype)
print(arr[:, -1])

## 2. 索引、切片、布尔索引

这部分后面会直接迁移到张量操作。  

常见用途

- 取部分特征列（selecting feature columns）
- 取某个 batch 的样本
- 按条件筛选样本（filtering samples by condition）

In [ ]:
matrix = np.arange(1, 17).reshape(4, 4)

print("matrix =")
print(matrix)
print()
print("前两列的中间两行 / middle two rows of first two columns:")
print(matrix[1:3, :2])
print()
print("最后一列 / last column:")
print(matrix[:, -1])
print()
mask = matrix % 2 == 0
print("布尔掩码 / boolean mask:")
print(mask)
print("所有偶数 / all even numbers:")
print(matrix[mask])

In [ ]:
# 练习 2
# 创建一个 5x5 数组，并完成
# 1. 取中心 3x3
# 2. 取第 0、2、4 行
# 3. 取所有大于 10 的元素
# 4. 按行反转

grid = np.arange(25).reshape(5, 5)

# center =
# picked_rows =
# bigger_than_10 =
# reversed_rows =

# print(grid)
# print(center)
# print(picked_rows)
# print(bigger_than_10)
# print(reversed_rows)

In [ ]:
# 练习 2 参考答案

grid = np.arange(25).reshape(5, 5)
center = grid[1:4, 1:4]
picked_rows = grid[[0, 2, 4]]
bigger_than_10 = grid[grid > 10]
reversed_rows = grid[::-1]

print(grid)
print(center)
print(picked_rows)
print(bigger_than_10)
print(reversed_rows)

## 3. 广播

`broadcasting` 是后面理解张量运算的关键。  

最小判断规则

- 从最后一个维度开始对齐
- 两个维度相等，或其中一个为 1，才可广播
- 否则会报错（otherwise the operation fails）

In [ ]:
features = np.array([
    [1.0, 10.0],
    [2.0, 20.0],
    [3.0, 30.0],
])

offset = np.array([100.0, 1000.0])
scale = np.array([[1.0], [10.0], [100.0]])

print("features + offset =")
print(features + offset)
print()
print("features * scale =")
print(features * scale)

要能用 `shape` 解释上面的结果。  

- `offset.shape == (2,)`，按最后一维对齐
- `scale.shape == (3, 1)`，可按行广播（broadcasts by row）

In [ ]:
# 练习 3
# 实现 center_by_column(x)，让每一列减去该列均值。
# Implement center_by_column(x) so each column is centered by its mean.

def center_by_column(x):
    # TODO
    pass


# x = np.array([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
# print(center_by_column(x))

In [ ]:
# 练习 3 参考答案

def center_by_column_solution(x):
    col_mean = x.mean(axis=0, keepdims=True)
    return x - col_mean


x = np.array([[1.0, 10.0], [2.0, 20.0], [3.0, 30.0]])
print(center_by_column_solution(x))

## 4. 向量化

在数值计算里，很多循环都可以改成数组表达式。  

Vectorization 的价值：

- 更简洁（shorter code）
- 更接近数学表达式（closer to the math）
- 通常更快（often faster）

In [ ]:
values = np.arange(1, 6)

loop_square_sum = 0.0
for value in values:
    loop_square_sum += value ** 2

vectorized_square_sum = (values ** 2).sum()

print("循环版本 / loop version:", loop_square_sum)
print("向量化版本 / vectorized version:", vectorized_square_sum)

In [ ]:
# 练习 4
# 不用 for 循环，实现 MSE
# mse = mean((pred - target) ** 2)

pred = np.array([2.5, 0.0, 2.0, 8.0])
target = np.array([3.0, -0.5, 2.0, 7.0])


def mse_vectorized(pred, target):
    # TODO
    pass


# print(mse_vectorized(pred, target))

In [ ]:
# 练习 4 参考答案

def mse_vectorized_solution(pred, target):
    return np.mean((pred - target) ** 2)


print(mse_vectorized_solution(pred, target))

## 5. 形状变换

你需要逐步建立这样的直觉：  

- 哪一步是在改形状（which step changes shape）
- 哪一步是在增加维度
- 哪一步是在重排轴顺序（which step reorders axes）

In [ ]:
flat = np.arange(24)
cube = flat.reshape(2, 3, 4)
transposed = cube.transpose(0, 2, 1)
expanded = np.arange(6).reshape(2, 3)[:, np.newaxis, :]

print("flat.shape =", flat.shape)
print("cube.shape =", cube.shape)
print("transposed.shape =", transposed.shape)
print("expanded.shape =", expanded.shape)

In [ ]:
# 练习 5
# 已知 flat_images 的 shape 是 (2, 12)，表示两张被展平的图片。
# flat_images has shape (2, 12), representing two flattened images.
# 假设每张图片原本是 (3, 4)
#
# 1. 还原成 (2, 3, 4)
# 2. 再转成 (2, 4, 3)

flat_images = np.arange(24).reshape(2, 12)

# images =
# images_t =

# print(images.shape)
# print(images_t.shape)

In [ ]:
# 练习 5 参考答案

flat_images = np.arange(24).reshape(2, 12)
images = flat_images.reshape(2, 3, 4)
images_t = images.transpose(0, 2, 1)

print(images.shape)
print(images_t.shape)

## 6. 小型 ML 场景

下面用一个极小的表格例子感受数值特征标准化。  


In [ ]:
# 列含义
raw = np.array([
    [1.5, 0.70, 6.0, 0],
    [3.0, 0.90, 7.0, 1],
    [2.2, 0.80, 6.5, 1],
    [1.0, 0.60, 5.5, 0],
], dtype=np.float32)

features = raw[:, :-1]
labels = raw[:, -1].astype(np.int64)

feature_mean = features.mean(axis=0, keepdims=True)
feature_std = features.std(axis=0, keepdims=True)
features_std = (features - feature_mean) / (feature_std + 1e-8)

print("features.shape =", features.shape)
print("labels.shape =", labels.shape)
print("标准化特征 / standardized features =")
print(features_std)

In [ ]:
# 练习 6
# 实现 split_xy_and_standardize(raw)
# Implement split_xy_and_standardize(raw)
#
# Input: 二维数组，最后一列是标签
# 输出

def split_xy_and_standardize(raw):
    # TODO
    pass


# x_std, y = split_xy_and_standardize(raw)
# print(x_std)
# print(y)

In [ ]:
# 练习 6 参考答案

def split_xy_and_standardize_solution(raw):
    x = raw[:, :-1]
    y = raw[:, -1].astype(np.int64)
    mean = x.mean(axis=0, keepdims=True)
    std = x.std(axis=0, keepdims=True)
    x_std = (x - mean) / (std + 1e-8)
    return x_std, y


x_std, y = split_xy_and_standardize_solution(raw)
print(x_std)
print(y)

## 7. 改错题

下面的代码会报广播错误。  

你要先判断：它是想按行乘，还是按列乘？  


In [ ]:
x = np.arange(12).reshape(3, 4)
weights = np.array([0.1, 0.2, 0.3])

# print(x * weights)
# TODO:
# 1. 写出按行乘的修复版本
# 2. 写出按列乘的修复版本

In [ ]:
# 改错题参考答案

x = np.arange(12).reshape(3, 4)
weights = np.array([0.1, 0.2, 0.3])

row_scaled = x * weights[:, np.newaxis]
print("按行缩放 / row-wise scaling =")
print(row_scaled)
print()

col_weights = np.array([0.1, 0.2, 0.3, 0.4])
col_scaled = x * col_weights
print("按列缩放 / column-wise scaling =")
print(col_scaled)

## 8. 小结

这一节最重要的是建立 `shape` 思维。  

你现在应该能回答

1. `shape`、`ndim`、`dtype` 各表示什么？
2. 布尔索引和普通切片有什么区别？
3. 广播的最小判断规则是什么？
4. 为什么向量化更适合数值计算？
5. `reshape` 和 `transpose` 有什么区别？

下一步建议

- 进入 `Pandas` notebook，把数组思维接到表格数据处理上